In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Google Colab rilevato: installazione di skrl e avvio di TensorBoard in corso...")
    
    # Installazione di skrl (usa subprocess per evitare problemi con i punti esclamativi negli 'if')
    import subprocess
    subprocess.run(["pip", "install", "-q", "skrl"])
    
    # Avvio di TensorBoard tramite le magie di IPython
    try:
        get_ipython().run_line_magic("load_ext", "tensorboard")
        get_ipython().run_line_magic("tensorboard", "--logdir runs")
    except Exception as e:
        print(f"Impossibile avviare automaticamente TensorBoard: {e}")
else:
    print("Ambiente locale rilevato: salto l'installazione e l'avvio di TensorBoard.")

Ambiente locale rilevato: salto l'installazione e l'avvio di TensorBoard.


In [15]:
from include.envs import SimpleRoTDetector

env = SimpleRoTDetector(n_tracks=100, spatial_res=0.1, time_res=0.3, tof_margin=50.0, debug=False)

In [16]:
from skrl.models.torch import Model, MultivariateGaussianMixin, DeterministicMixin
from torch import nn
import torch


class Actor(MultivariateGaussianMixin, Model):
    def __init__(self, observation_space, action_space, device):
        Model.__init__(self, observation_space=observation_space, action_space=action_space, device=device)
        MultivariateGaussianMixin.__init__(self, clip_actions=True, min_log_std=-20, max_log_std=2)

        self.net = nn.Sequential(
            nn.Linear(self.num_observations, 2**10), nn.Sigmoid(),
            nn.Linear(2**10, 2**12), nn.Sigmoid(),
            nn.Linear(2**12, 2**10), nn.Sigmoid(),
            nn.Linear(2**10, self.num_actions) 
        )

        self.log_std_parameter = nn.Parameter(torch.zeros(self.num_actions))

    def compute(self, inputs, role=""):
        mean_actions = self.net(inputs["observations"])

        rpo_alpha = inputs["alpha"]
        perturbation = torch.zeros_like(mean_actions).uniform_(-rpo_alpha, rpo_alpha)
        mean_actions += perturbation

        return mean_actions, {"log_std": self.log_std_parameter}


class Critic(DeterministicMixin, Model):
    def __init__(self, observation_space, action_space, device):
        Model.__init__(self, observation_space=observation_space, action_space=action_space, device=device)
        DeterministicMixin.__init__(self, clip_actions=False)

        self.net = nn.Sequential(
            nn.Linear(self.num_observations, 2**12), nn.Sigmoid(),
            nn.Linear(2**12, 2**10), nn.Sigmoid(),
            nn.Linear(2**10, 2**8), nn.Sigmoid(),
            nn.Linear(2**8, 1),
        )

    def compute(self, inputs, role=""):
        x = inputs["observations"]
        return self.net(x), {}


In [17]:
torch.autograd.set_detect_anomaly(True)

In [18]:
from skrl.envs.wrappers.torch import wrap_env
from skrl.memories.torch import RandomMemory
from skrl.resources.noises.torch import GaussianNoise
from skrl.agents.torch.rpo import RPO, RPO_CFG
from skrl.resources.preprocessors.torch import RunningStandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

env = wrap_env(env, wrapper = 'gymnasium')
memory = RandomMemory(memory_size=2048, num_envs=env.num_envs, device=device)

models = {
    "policy": Actor(env.observation_space, env.action_space, device),
    "value": Critic(env.observation_space, env.action_space, device),
}

cfg = RPO_CFG(
    mini_batches = 8,
    discount_factor=0.98,
    learning_rate=1e-4,
    rollouts=2048
)
cfg.experiment.write_interval = 20
cfg.experiment.checkpoint_interval = 0
cfg.experiment.experiment_name = "RPO-RoT"
cfg.experiment.directory = "./runs"
cfg.close_environment_at_exit = True
cfg.timesteps = 50000
cfg.disable_progressbar = False
cfg.headless = True
cfg.environment_info = True
cfg.state_preprocessor = RunningStandardScaler
cfg.state_preprocessor_kwargs = {"size": env.observation_space, "device": device}
cfg.value_preprocessor = RunningStandardScaler
cfg.value_preprocessor_kwargs = {"size": 1, "device": device}
cfg.experiment.checkpoint_interval = "auto"

agent = RPO(
    models=models,
    memory=memory,
    observation_space=env.observation_space,
    action_space=env.action_space,
    device=device,
    cfg=cfg,
)



[skrl:INFO] Environment wrapper: Gymnasium
[skrl:WARNING] Failed to check for a vectorized environment: module 'gymnasium.experimental' has no attribute 'vector'


In [ ]:
from skrl.trainers.torch import SequentialTrainer

trainer = SequentialTrainer(cfg=cfg, env=env, agents=agent)
trainer.train()
